# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruchitgoud/flyrankai-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use a Decision Tree because it fits the content review-priority lane and is easy to interpret. The model can learn simple decision patterns from observable content and performance signals and can show which signals are useful for prioritizing pages for review.

This is appropriate for this task because the goal is decision support: identify content items that may deserve review or refresh. I will compare the model against the Week-4 rule-based baseline using the same data and evaluation metric.

The model is not intended to prove that refreshing a page will cause better performance. It is intended to provide a measurable and interpretable prioritization signal.

In [2]:
import os
import pandas as pd

repo_path = "/content/flyrankai-intern"

# Clone the repository if it is not already available
if not os.path.exists(repo_path):
    !git clone https://github.com/ruchitgoud/flyrankai-intern.git

# Move into the repository
%cd /content/flyrankai-intern

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nTarget values:")
print(df["trend_direction"].value_counts(dropna=False))

Cloning into 'flyrankai-intern'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 162 (delta 68), reused 95 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 1.87 MiB | 9.76 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/flyrankai-intern
Dataset shape: (30000, 44)

Target values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a time-aware split so that the model is evaluated on data that represents a later decision period than the training data.

The target will be the `trend_direction` proxy, with `down` treated as the positive class for this modeling exercise. I will use only observable features available at the decision point and exclude fields that directly encode the target or future outcomes.

The same evaluation data and metric will be used when comparing the model with the Week-4 baseline, so the comparison is fair.

In [3]:
from sklearn.model_selection import train_test_split

# 1 = declining, 0 = not declining
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nTarget distribution — training:")
print(train_df["is_declining_label"].value_counts(normalize=True))

print("\nTarget distribution — test:")
print(test_df["is_declining_label"].value_counts(normalize=True))

Training rows: 26581
Test rows: 3419

Training clients: 25
Test clients: 7

Target distribution — training:
is_declining_label
1    0.544411
0    0.455589
Name: proportion, dtype: float64

Target distribution — test:
is_declining_label
1    0.523837
0    0.476163
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model training and comparison

I will train a Decision Tree using observable content and performance features available at the decision point.

I will exclude `trend_direction` and other target-derived or future-looking fields from the model inputs. The model will be evaluated on the held-out client set.

I will compare the model with a simple baseline using the same test records and evaluation metrics. This comparison is intended to measure whether the learned model provides additional predictive value over the rule-based baseline.

In [4]:
# Section 3 — Train model and compare with baseline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

features = numeric_features + categorical_features

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

model_prob = pipeline.predict_proba(X_test)[:, 1]

model_auc = roc_auc_score(y_test, model_prob)
model_ap = average_precision_score(y_test, model_prob)

print("Decision Tree ROC AUC:", round(model_auc, 4))
print("Decision Tree Average Precision:", round(model_ap, 4))

baseline_test = test_df.copy()

staleness_score = (
    baseline_test["days_since_last_update"]
    .rank(pct=True)
    .fillna(0)
)

baseline_test["position_bucket"] = pd.cut(
    baseline_test["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_median_ctr = (
    baseline_test.groupby(
        "position_bucket",
        observed=False
    )["ctr"]
    .transform("median")
)

ctr_gap = (
    position_median_ctr - baseline_test["ctr"]
).clip(lower=0)

ctr_gap_score = (
    ctr_gap.rank(pct=True)
    .fillna(0)
)

baseline_score = (
    0.60 * staleness_score
    + 0.40 * ctr_gap_score
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_score
)

baseline_ap = average_precision_score(
    y_test,
    baseline_score
)

print("\nWeek-4 Baseline ROC AUC:", round(baseline_auc, 4))
print("Week-4 Baseline Average Precision:", round(baseline_ap, 4))

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Week-5 Decision Tree"
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ],
    "average_precision": [
        baseline_ap,
        model_ap
    ]
})

display(comparison)

Decision Tree ROC AUC: 0.6392
Decision Tree Average Precision: 0.6105

Week-4 Baseline ROC AUC: 0.5104
Week-4 Baseline Average Precision: 0.5207


,method,roc_auc,average_precision
0,Week-4 baseline,0.510426,0.520683
1,Week-5 Decision Tree,0.639220,0.610493


### Comparison result

The Week-5 Decision Tree outperformed the Week-4 baseline on the held-out client set.

The baseline achieved a ROC AUC of 0.5104 and average precision of 0.5207, while the Decision Tree achieved a ROC AUC of 0.6392 and average precision of 0.6105.

This suggests that the learned model provides a stronger predictive signal for the `down` proxy label than the simple Week-4 rule under this evaluation setup. However, these results do not prove that refreshing a page will cause improved performance. The model should still be treated as decision support and validated on future data before operational use.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Decision Tree performs better than the Week-4 baseline on the held-out client set, but it still makes incorrect predictions.

I will inspect false positives and false negatives to understand where the model is uncertain or may be misleading. I will also examine feature importance to understand which observable signals the tree relies on most.

These errors are important because a false positive could cause a page to receive unnecessary review attention, while a false negative could cause a potentially important declining page to be missed.

The model should therefore be treated as a prioritization and decision-support tool rather than proof that a page needs to be refreshed. The observed performance is specific to this evaluation setup and proxy label.

In [5]:
# Section 4 — Errors and interpretation

from sklearn.metrics import confusion_matrix

model_pred = (model_prob >= 0.50).astype(int)

error_df = test_df[
    [
        "content_id",
        "client_id",
        "is_declining_label",
        "trend_direction",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
].copy()

error_df["predicted_probability"] = model_prob
error_df["predicted_label"] = model_pred

false_positives = error_df[
    (error_df["is_declining_label"] == 0) &
    (error_df["predicted_label"] == 1)
].copy()

false_negatives = error_df[
    (error_df["is_declining_label"] == 1) &
    (error_df["predicted_label"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(10))

print("\nExample false negatives:")
display(false_negatives.head(10))

feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = pipeline.named_steps[
    "model"
].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop 15 model features:")
display(feature_importance.head(15))

False positives: 596
False negatives: 689

Example false positives:


,content_id,client_id,is_declining_label,trend_direction,days_since_last_update,ctr,avg_position,impressions_90d,predicted_probability,predicted_label
13,content_a5a2fbc76336,client_8527a891e2,0,stable,103,0.00,39.8,307,0.821263,1
100,content_cc97c093c379,client_a88a7902cb,0,stable,20,3.26,11.9,184,0.648704,1
170,content_33bbf9c88e69,client_9f14025af0,0,up,20,0.00,33.9,429,0.648704,1
179,content_552a9396d8dc,client_8527a891e2,0,up,104,0.72,12.9,416,0.648704,1
222,content_cdeaa91ddaa5,client_a88a7902cb,0,stable,20,0.09,39.1,1084,0.648704,1
261,content_b6f1aab067b3,client_9400f1b21c,0,stable,20,0.08,11.3,4895,0.524963,1
428,content_3b450f52f8c6,client_bbb965ab0c,0,up,20,0.43,32.5,235,0.648704,1
432,content_912dddcb7453,client_bbb965ab0c,0,up,8,0.24,65.3,416,0.648704,1
448,content_410697dea3ca,client_bbb965ab0c,0,stable,15,0.13,6.2,18769,0.648704,1
490,content_895ce765d29d,client_8527a891e2,0,stable,104,0.51,29.7,197,0.648704,1



Example false negatives:


,content_id,client_id,is_declining_label,trend_direction,days_since_last_update,ctr,avg_position,impressions_90d,predicted_probability,predicted_label
39,content_4595e8704e07,client_8527a891e2,1,down,104,0.00,36.3,4,0.082186,0
44,content_793b7376a0e5,client_8527a891e2,1,down,20,0.00,55.8,64,0.478811,0
49,content_f0717373e86e,client_8527a891e2,1,down,8,0.00,10.1,9,0.478811,0
68,content_6bc2ec5f6061,client_a88a7902cb,1,down,8,0.81,23.0,124,0.064854,0
113,content_824ddd97c0ae,client_bbb965ab0c,1,down,20,3.85,6.5,26,0.478811,0
160,content_a62b745f8be3,client_8527a891e2,1,down,20,0.00,32.5,15,0.478811,0
213,content_17e8b736fca7,client_8527a891e2,1,down,20,0.00,9.0,36,0.478811,0
220,content_474bc8a4a3cb,client_8527a891e2,1,down,98,0.00,5.7,3,0.306821,0
238,content_b9b869d9cb84,client_8527a891e2,1,down,20,0.00,80.0,42,0.478811,0
272,content_5eabfc22996f,client_8527a891e2,1,down,104,0.00,10.3,50,0.478811,0



Top 15 model features:


,feature,importance
9,num__days_with_impressions,0.486330
11,num__content_age_days,0.209504
6,num__clicks_90d,0.078665
14,num__avg_position,0.067878
16,num__scroll_rate,0.051482
36,cat__word_count_tier_1000-2000,0.034913
5,num__impressions_90d,0.027076
10,num__days_with_sessions,0.021212
12,num__days_since_last_update,0.017918
1,num__competition,0.001899


### Interpretation

The Decision Tree performed better than the Week-4 baseline on the held-out client set, but the model still produces both false positives and false negatives.

False positives represent pages that the model would prioritize even though they are not in the `down` proxy class. These could lead to unnecessary review effort. False negatives represent declining pages that the model fails to prioritize, which could cause useful review opportunities to be missed.

The feature-importance results help explain which observable signals the model relies on, but feature importance does not establish causation. The model's performance is specific to this 30,000-row starter dataset, the client-holdout validation design, and the `trend_direction == "down"` proxy label.

The model is therefore best treated as decision support for prioritizing pages for human review, rather than as proof that a page needs a refresh or that a refresh will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.